In [2]:
import polars as pl

#root_dir = "/home/ayesh/Documents/MSc/Period1/Introduction to Data Science/MiniProject/weather-driven-transit-delay-forecasting/"
df = pl.scan_parquet(["../../data/interim/combined_actual_delays.parquet"])
df.head(5).collect()

datetime,trip_id,stop_sequence,stop_id,actual_arrival_delay,arrival_uncertainty,actual_departure_delay,departure_uncertainty,timestamp
str,i64,i64,i64,f64,f64,f64,f64,i64
"""2022-06-29 21:59:54""",14010000607402196,1,9022001055075002,47.0,null,47.0,0.0,1656539994
"""2022-06-29 21:59:54""",14010000612481735,1,9022001011725019,0.0,null,7.0,0.0,1656539994
"""2022-06-29 21:59:54""",14010000611286946,10,9022001007531001,8.0,0.0,72.0,0.0,1656539994
"""2022-06-29 21:59:54""",14010000610964728,8,9022001051054001,162.0,0.0,162.0,0.0,1656539994
"""2022-06-29 21:59:54""",14010000610964619,12,9022001050020001,-17.0,null,-17.0,null,1656539994


In [27]:
# Number of rows
df.select(pl.len()).collect()

len
u32
433845292


In [23]:
import duckdb

result = duckdb.sql("""
SELECT COUNT(*) AS unique_pairs
FROM (
    SELECT DISTINCT trip_id, stop_id
    FROM '../..//data/interim/combined_actual_delays.parquet'
)
""").fetchone()[0]

print("Unique (trip_id, stop_id) pairs:", result)


Unique (trip_id, stop_id) pairs: 10006452


In [50]:
# Drop irrelevant columns
df = df.drop(["stop_sequence", "arrival_uncertainty", "actual_departure_delay", "departure_uncertainty",	"timestamp"])

# Process datetime column and create hour of the day(hod) column
df = (
    df.with_columns(
        pl.col("datetime").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
    )
    .with_columns([
        pl.col("datetime").cast(pl.Date).alias("date"),   # actual Date type
        pl.col("datetime").dt.hour().alias("HOD"),        # hour of day
    ])
    .drop("datetime")
)
df = df.rename({'actual_arrival_delay': 'delay'})

In [51]:
# Checking missing values
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'trip_id': 552995, 'stop_id': 0, 'delay': 490858, 'date': 0, 'HOD': 0}


In [52]:
# Drop rows with missing trip_id as they cannot be combined 
# with GTFS-Static data

df = df.drop_nulls(subset=["trip_id"])

# Checking missing values
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'trip_id': 0, 'stop_id': 0, 'delay': 483209, 'date': 0, 'HOD': 0}


In [53]:
# Write out lazily to Parquet (streaming-safe)
df.sink_parquet("../../data/interim/gtfs_rt_parquet/actual_delays_cleaned.parquet")

In [69]:
df = pl.scan_parquet(["../../data/interim/gtfs_rt_parquet/actual_delays_cleaned.parquet"])
df.head(5).collect()

trip_id,stop_id,delay,date,HOD
i64,i64,f64,date,i8
14010000607402196,9022001055075002,47.0,2022-06-29,21
14010000612481735,9022001011725019,0.0,2022-06-29,21
14010000611286946,9022001007531001,8.0,2022-06-29,21
14010000610964728,9022001051054001,162.0,2022-06-29,21
14010000610964619,9022001050020001,-17.0,2022-06-29,21


In [64]:
tdf = df.with_columns([
    pl.col("date").dt.month().alias("month")
])

# Count rows for May–October
may_oct_count = (
    tdf.filter(pl.col("month").is_in([5,6,7,8,9,10]))
      .select(pl.len().alias("count"))
      .collect()
)["count"][0]

# Count rows for Nov–Apr
nov_apr_count = (
    tdf.filter(pl.col("month").is_in([11,12,1,2,3,4]))
      .select(pl.len().alias("count"))
      .collect()
)["count"][0]

print(f"Rows in May–Oct: {may_oct_count}")
print(f"Rows in Nov–Apr: {nov_apr_count}")

Rows in May–Oct: 221585785
Rows in Nov–Apr: 211706512


In [71]:
# Keep only rows outside May–October (Snowing months)
df_filtered = (
    df.lazy()
      .filter(
          ~pl.col("date")
            .dt.month()
            .is_in([5,6,7,8,9,10])
      )
)

# Write out lazily to Parquet (streaming-safe)
df_filtered.sink_parquet("../../data/interim/gtfs_rt_parquet/gtfs_rt_snow_filtered.parquet")


In [72]:
df = pl.scan_parquet(["../../data/interim/gtfs_rt_parquet/gtfs_rt_snow_filtered.parquet"])
df.select(pl.len()).collect()

len
u32
211706512


In [73]:
tdf = df.with_columns([
    pl.col("date").dt.month().alias("month")
])

# Count rows for May–October
may_oct_count = (
    tdf.filter(pl.col("month").is_in([5,6,7,8,9,10]))
      .select(pl.len().alias("count"))
      .collect()
)["count"][0]

# Count rows for Nov–Apr
nov_apr_count = (
    tdf.filter(pl.col("month").is_in([11,12,1,2,3,4]))
      .select(pl.len().alias("count"))
      .collect()
)["count"][0]

print(f"Rows in May–Oct: {may_oct_count}")
print(f"Rows in Nov–Apr: {nov_apr_count}")

Rows in May–Oct: 0
Rows in Nov–Apr: 211706512


In [36]:
df = pl.scan_parquet(["../../data/interim/weather/weather.parquet"])
df.filter(pl.col("Date") >= "2022-01-01").head().collect()

Date,Temperature,Precipitation,SnowDepth,WindSpeed,HOD
str,f64,f64,f64,f64,i64
"""2022-01-01""",0.9,0.0,0.0,null,0
"""2022-01-01""",0.3,0.0,0.0,null,1
"""2022-01-01""",0.2,0.2,0.0,null,2
"""2022-01-01""",0.6,0.1,0.0,null,3
"""2022-01-01""",0.4,0.0,0.0,null,4


In [32]:
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'Date': 0, 'Temperature': 0, 'Precipitation': 0, 'SnowDepth': 0, 'WindSpeed': 38807, 'HOD': 0}


In [33]:
wind_df = pl.scan_parquet(["../../data/interim/weather/wind_data.parquet"])
wind_df.tail().collect()

Date,hod,Wind Speed(m/s)
str,i64,f64
"""6/30/2025""",19,2.1
"""6/30/2025""",20,3.8
"""6/30/2025""",21,2.4
"""6/30/2025""",22,0.6
"""6/30/2025""",23,0.0


In [34]:
import polars as pl

df = df.with_columns(
    pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d", strict=False),
)

wind_df = (
    wind_df.rename({
        "hod": "HOD",
        "Wind Speed(m/s)": "WindSpeed_new",
    })
    .with_columns(
        pl.col("Date").str.strptime(pl.Date, "%m/%d/%Y", strict=False),
    )
)

# 2️⃣ Right join on Date + HOD
joined_lf = df.join(
    wind_df,
    on=["Date", "HOD"],
    how="right",  # keep all from wind_lf
)

# 3️⃣ Replace old WindSpeed with new WindSpeed_new (prefer wind data when available)
joined_lf = joined_lf.with_columns(
    pl.coalesce(pl.col("WindSpeed_new"), pl.col("WindSpeed")).alias("WindSpeed")
).drop("WindSpeed_new")

joined_lf.tail().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
14.4,0.0,0.0,2.1,2025-06-30,19
12.4,0.0,0.0,3.8,2025-06-30,20
10.5,0.0,0.0,2.4,2025-06-30,21
9.3,0.0,0.0,0.6,2025-06-30,22
null,null,null,0.0,2025-06-30,23


In [41]:
joined_lf.head().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
0.9,0.0,0.0,5.1,2022-01-01,0
0.3,0.0,0.0,3.4,2022-01-01,1
0.2,0.2,0.0,3.2,2022-01-01,2
0.6,0.1,0.0,4.9,2022-01-01,3
0.4,0.0,0.0,4.9,2022-01-01,4


In [44]:
joined_lf.sink_parquet("../../data/interim/weather/merged_weather.parquet")  # streaming-safe
merged_df = pl.scan_parquet(["../../data/interim/weather/merged_weather.parquet"])
merged_df.head().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
-2.4,0.0,0.0,1.5,2022-01-01,20
-2.0,0.0,0.0,2.6,2022-01-01,22
0.0,1.3,2.848,3.4,2022-01-02,2
1.6,0.0,0.0,2.0,2022-01-02,8
2.5,0.0,0.0,1.2,2022-01-02,12


In [49]:
merged_df.sort("Date", "HOD").head().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
0.9,0.0,0.0,5.1,2022-01-01,0
0.3,0.0,0.0,3.4,2022-01-01,1
0.2,0.2,0.0,3.2,2022-01-01,2
0.6,0.1,0.0,4.9,2022-01-01,3
0.4,0.0,0.0,4.9,2022-01-01,4
